In [1]:
from vertex_lite.custom import load_data,load_data_v3,load_data_v3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
import glob
import os
import numpy as np

In [2]:
dftest=load_data_v3('resultados_cluster/noise_0.1/pct_0.0/sim_0/cells.parquet',columns=['time','n_lados','neighbor_of_mutant','cell_id','alive'],time_filter=1000)
dftest=dftest[dftest['neighbor_of_mutant'] == True]

dftest

,time,n_lados,neighbor_of_mutant,cell_id,alive,mesh_type,pct_mutant,sim_id


## Load Sort and Filter

In [3]:
def filter_data(time,local=False,cluster=False):
    if cluster:
        rpath='resultados_cluster'
    else:
        rpath='resultados'
    
    df=load_data(f'{rpath}/noise_*/pct_*/sim_*/cells.parquet',columns=['time','n_lados','neighbor_of_mutant','cell_id','alive'],time_filter=time)
    if df.empty:
        return print(f"No data found for time {time} and local {local}")
    df = df[df['alive'] == 1]
    if local:
        df=df[df['neighbor_of_mutant'] == True]
        
        
    else:
        df=df.drop(columns=['neighbor_of_mutant'])

    df = (
        df[df['n_lados'] >= 3]
        .groupby(['time','mesh_type','pct_mutant','n_lados'])
        .size()
        .reset_index(name='n_cells')
    )
    df['total_cells'] = (
        df.groupby(['time','mesh_type','pct_mutant'])['n_cells']
        .transform('sum')
    )
    df['frac_cells'] = df['n_cells'] / df['total_cells']
    return df


## Plotting

In [4]:
#plot_heatmap(time, local, "hex")
from fileinput import filename


def get_heatmap_data(time, local=False, case=None,cluster=False):
    
    if case is None:
        raise ValueError("case must be: hex, mean, weigh, std")
    df = filter_data(time, local, cluster)
    grouped = df.groupby(["mesh_type", "pct_mutant"])[['n_lados', 'n_cells', 'frac_cells']]

    if case == "hex":
        title = "Mean Absolute Deviation from Hexagon"

        def mad_hex(group):
            N = group["n_cells"].sum()
            return (group["n_cells"] * abs(group["n_lados"] - 6)).sum() / N
        
        heatmap_data = grouped.apply(mad_hex).unstack()
        

    elif case == "mean":
        title = "Mean Absolute Deviation from Mean"

        def mad_mean(group):
            N = group["n_cells"].sum()
            mu = (group["n_lados"] * group["n_cells"]).sum() / N
            return (group["n_cells"] * abs(group["n_lados"] - mu)).sum() / N

        heatmap_data = grouped.apply(mad_mean).unstack()

    elif case == "weigh":
        title = "Deviation from Hexagon (using frac_cells)"

        def mad_frac(group):
            return (group["frac_cells"] * abs(group["n_lados"] - 6)).sum()

        heatmap_data = grouped.apply(mad_frac).unstack()

    elif case == "std":
        title = "Weighted Standard Deviation"

        def weighted_std(group):
            N = group["n_cells"].sum()
            mu = (group["n_lados"] * group["n_cells"]).sum() / N
            var = (group["n_cells"] * (group["n_lados"] - mu) ** 2).sum() / N
            return np.sqrt(var)

        heatmap_data = grouped.apply(weighted_std).unstack()

    else:
        raise ValueError("case must be: hex, mean, weigh, std")
    heatmap_data = heatmap_data.sort_index().sort_index(axis=1)
    heatmap_data = heatmap_data.T

    if local and 0.0 not in heatmap_data.index:
        heatmap_data.loc[0.0] = 0

    heatmap_data.index = pd.to_numeric(heatmap_data.index)
    heatmap_data = heatmap_data.sort_index()

    return heatmap_data, title

def plot_heatmap(heatmap_data,title,time, local=False, case=None,cluster=False):
    cmap = sns.light_palette("#EA5A00", as_cmap = True)
    plt.figure(figsize=(7, 5))
    sns.set_theme(style="white", context="talk")
    # cmap = sns.color_palette("ch:s=.3,r=.1", as_cmap=True)

    plt.rcParams.update({
        "font.size": 20,
        "axes.labelsize": 25,
        "axes.titlesize": 25,
        "legend.fontsize": 12,
        "axes.linewidth": 0.0,
        "xtick.labelsize": 24,
        "ytick.labelsize": 24,
    })
    # sns.heatmap(

    #     heatmap_data,
    #     annot=False,          # muestra los valores
    #     fmt=".3f",           # formato decimal
    #     cmap=cmap,
    #     linewidths=0,
    #     linecolor=None,
    #      square=False,
    #     vmin=0, 
    #     vmax=1.0,
    #     cbar_kws={"label": case},
    #     ax=ax
    # )
    # ax.annotate(
    #     f'T={time} | local={local}',
    #     xy=(0.0, -0.2),
    #     xycoords='axes fraction',
    #     ha='left',
    #     fontsize=10
    # )
    # ax.invert_yaxis()  # Para que el primer mesh_type esté en la parte superior
    # ---- PLOT WITH IMSHOW ----
    fig, ax = plt.subplots(figsize=(8, 6))

    # Convert DataFrame to numpy array for imshow
    # Note: imshow handles the orientation differently than sns, 
    # so we use the transposed/unstacked data directly.
    im = ax.imshow(
        heatmap_data.values, 
        cmap=cmap, 
        origin='lower',  # Puts the first row at the bottom (standard for plots)
        aspect='auto',   # 'equal' makes cells square, 'auto' fills the figure
        vmin=0, 
        vmax=1.0
    )

    # --- AXES LABELS ---
    # In imshow, we must manually set the ticks and labels
    step = 2  
    plt.tick_params(axis='both', labelsize = 15)
    # X-axis: Mesh Distortion (Columns)
    x_ticks = np.arange(0, len(heatmap_data.columns), step)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels(heatmap_data.columns[::step], rotation=45)

    # Y-axis: % mutants (Index)
    y_ticks = np.arange(0, len(heatmap_data.index), step)
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(heatmap_data.index[::step])

    # --- COLORBAR & LABELS ---
  
    # --- COLORBAR ---
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label(case)

    # Annotations and Titles
    ax.annotate(
        f'T={time} | local={local}',
        xy=(0.0, -0.2),
        xycoords='axes fraction',
        ha='left',
        fontsize=10
    )

    # Labels (Standard Matplotlib)
    plt.xlabel("Initial Mesh Distortion")
    plt.ylabel("% mutants")
    plt.title(f'{title} for Neighbors of Mutant Cells' if local else f'{title} for Whole Tissue')
    plt.xlabel("Initial Mesh Distortion")
    plt.ylabel("% mutants")
    plt.title(f'{title} for Neighbors of Mutant Cells' if local else f'{title} for Whole Tissue')

    plt.tight_layout()
    
    suffix = "local" if local else "global"
    rpath='resultados_cluster' if cluster else 'resultados'
    clustering= "cluster" if cluster else "random"
    
    savedir = f"{rpath}/figures/paper/nlados2_heatmaps/{case}"
    savedir = "figures_paper/pol_heatmaps_2"
    os.makedirs(savedir, exist_ok=True)

    filename = f"{savedir}/heatmap_{clustering}_time_{time}_{case}_{suffix}.svg"

    fig.savefig(filename, format ='svg')
    plt.close(fig)

## Figures

In [5]:
# times=[[0,100,250,1995], [0,1000,2500,4000]]
# times=[[4995], [ 4995]]
times= [[1995], [14995]]
# for c in ["hex",'std']:
#     for l in [True, False]:
#         for t in times[0]:
#             plot_heatmap_v2(time=t, local=l, case=c,cluster=False)
#         for t in times[1]:
#             plot_heatmap_v2(time=t, local=l, case=c,cluster=True)


In [6]:
data_cache = {}

# for c in ["hex", "std"]:
for c in ["hex"]: # we are only going to use the deviation from the hexagon
    for l in [True, False]:
        # Process times for local
        for t in times[0]:
            key = (t, l, c, False) # (time, local, case, cluster)
            print(f"Analyzing {key}...")
            data_cache[key] = get_heatmap_data(time=t, local=l, case=c, cluster=False)
            
        # Process times for cluster
        for t in times[1]:
            key = (t, l, c, True)
            print(f"Analyzing {key}...")
            data_cache[key] = get_heatmap_data(time=t, local=l, case=c, cluster=True)

print("✅ All data loaded into memory.")

Analyzing (1995, True, 'hex', False)...
Analyzing (14995, True, 'hex', True)...
Analyzing (1995, False, 'hex', False)...
Analyzing (14995, False, 'hex', True)...
✅ All data loaded into memory.


In [7]:
# Change these variables to tweak the look of all plots instantly!
CUSTOM_STEP = 1 
SHOW_SAVE = True

for key, (heatmap_df, title) in data_cache.items():
    t, l, c, is_cluster = key
    
    # Call the fast rendering function
    plot_heatmap(heatmap_df,title,t, local=l, case=c,cluster=is_cluster)
    print(l)
    # If working in a notebook, you might want to remove plt.close 
    # to see them all, or keep it to save memory.
    # plt.close()

True
True
False
False


<Figure size 700x500 with 0 Axes>

<Figure size 700x500 with 0 Axes>

<Figure size 700x500 with 0 Axes>

<Figure size 700x500 with 0 Axes>